In [1]:
import sys
import warnings
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm
from scipy.optimize import minimize
sys.path.insert(0, "../../Utilities/")
import common_functions as cf

warnings.filterwarnings('ignore')

In [2]:
data_in = np.load('initial_inputs.npy')
data_out = np.load('initial_outputs.npy')
print(data_in)
print(data_out)

[[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]]
[6.44434399e+01 1.83013796e+01 1.12939795e-01 4.21089813e+0

Week-01

In [3]:
X_init = np.array([
[0.19144708, 0.03819337, 0.60741781, 0.41458414],
[0.75865295, 0.53651774, 0.65600038, 0.36034155],
[0.43834987, 0.8043397 , 0.21024527, 0.15129482],
[0.70605083, 0.53419196, 0.26424335, 0.48208755],
[0.83647799, 0.19360965, 0.6638927 , 0.78564888],
[0.68343225, 0.11866264, 0.82904591, 0.56757661],
[0.55362148, 0.66734998, 0.32380582, 0.81486975],
[0.35235627, 0.32224153, 0.11697937, 0.47311252],
[0.15378571, 0.72938169, 0.42259844, 0.44307417],
[0.46344227, 0.63002451, 0.10790646, 0.9576439 ],
[0.67749115, 0.35850951, 0.47959222, 0.07288048],
[0.58397341, 0.14724265, 0.34809746, 0.42861465],
[0.30688872, 0.31687813, 0.62263448, 0.09539906],
[0.51114177, 0.817957  , 0.72871042, 0.11235362],
[0.43893338, 0.77409176, 0.37816709, 0.93369621],
[0.22418902, 0.84648049, 0.87948418, 0.87851568],
[0.72526172, 0.47987049, 0.08894684, 0.75976022],
[0.35548161, 0.63961937, 0.41761768, 0.12260384],
[0.11987923, 0.86254031, 0.64333133, 0.84980383],
[0.12688467, 0.15342962, 0.77016219, 0.19051811]
])

y_init = np.array([
6.44434399e+01, 1.83013796e+01, 1.12939795e-01, 4.21089813e+00,
2.58370525e+02, 7.84343889e+01, 5.75715369e+01, 1.09571876e+02,
8.84799176e+00, 2.33223610e+02, 2.44230883e+01, 6.44201468e+01,
6.34767158e+01, 7.97291299e+01, 3.55806818e+02, 1.08885962e+03,
2.88667516e+01, 4.51815703e+01, 4.31612757e+02, 9.97233189e+00
])

X_init = data_in
y_init = data_out


kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_init, y_init)

# Surrogate to maximise (negative for minimize)
def surrogate_neg(x):
    return -gp.predict(x.reshape(1, -1))[0]

# Bounds for normalized inputs
bounds = [(0,1), (0,1), (0,1), (0,1)]

# Try multiple random starts to avoid local issues
best_x = None
best_val = float('inf')
for _ in range(10):
    x0 = np.random.rand(4)
    res = minimize(surrogate_neg, x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_val:
        best_val = res.fun
        best_x = res.x

x_next = best_x
print("Next point to evaluate:", x_next)


Next point to evaluate: [0.38062608 0.265779   0.78518333 0.79560751]


Week-02

In [4]:
X_init = data_in           # shape (n_samples, 4)
y_init = data_out          # shape (n_samples,)

# --- New evaluated point ---
x_new = np.array([0.232877,0.841416,0.883342,0.879464])
y_new = 1091.3271430129832

# --- Add new data to dataset ---
X_all = np.vstack([X_init, x_new])
y_all = np.hstack([y_init, y_new])

# --- Refit Gaussian Process ---
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_all, y_all)

# --- Generate candidate next points around current best ---
best_x = x_new
sigma = 0.02  # tweak size for local exploration
num_candidates = 5

x_next_candidates = best_x + np.random.normal(0, sigma, size=(num_candidates, 4))
# Ensure all points are within [0,1]
x_next_candidates = np.clip(x_next_candidates, 0, 1)

print("Candidate next points to evaluate:")
print(x_next_candidates)

Candidate next points to evaluate:
[[0.27159827 0.80951161 0.89221815 0.88856966]
 [0.24091438 0.84169049 0.86609699 0.92331168]
 [0.24267057 0.86412367 0.94055076 0.9094032 ]
 [0.19811026 0.82566708 0.84598075 0.86496227]
 [0.22112073 0.82285244 0.90937637 0.86692545]]


Week-03

In [5]:
X_init = data_in           # shape (n_samples, 4)
y_init = data_out          # shape (n_samples,)

# --- Add the two new data points ---
new_points = np.array([
    [0.232877,0.841416,0.883342,0.879464],
    [0.245154,0.843081,0.898729,0.883391]
])

new_outputs = np.array([
    1091.3271430129832,
    1195.7279770056589
])


# Combine all data
X_all = np.vstack([X_init, new_points])
y_all = np.hstack([y_init, new_outputs])

# --- Fit updated Gaussian Process ---
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-8,                # smaller noise term for precision
    normalize_y=True,
    n_restarts_optimizer=5     # more robust kernel fitting
)
gp.fit(X_all, y_all)

# --- Define acquisition function (UCB variant) ---
def surrogate_neg_ucb(x, kappa=2.0):
    mean, std = gp.predict(x.reshape(1, -1), return_std=True)
    return -(mean + kappa * std)

# --- Search bounds for normalized inputs ---
bounds = [(0, 1), (0, 1), (0, 1), (0, 1)]

# --- Optimize acquisition function for next sampling point ---
best_x, best_val = None, float('inf')
for _ in range(10):
    x0 = np.random.rand(4)
    res = minimize(surrogate_neg_ucb, x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_val:
        best_val = res.fun
        best_x = res.x

x_next = best_x

print("Suggested next point to evaluate:", x_next)

# --- Optionally: generate a few local perturbations for fine exploration ---
sigma = 0.01
num_candidates = 5
x_next_candidates = x_next + np.random.normal(0, sigma, size=(num_candidates, 4))
x_next_candidates = np.clip(x_next_candidates, 0, 1)

print("\nLocal candidate points for fine-tuning:")
print(x_next_candidates)

res_formatted = [f"{r:.6f}" for r in x_next]
result = "-".join(res_formatted)
print(result)

x_next_6dp = np.round(x_next, 6)
x_next_6dp

Suggested next point to evaluate: [0.07240794 0.49205946 0.20464139 0.69659391]

Local candidate points for fine-tuning:
[[0.07224986 0.47694456 0.21949093 0.70445449]
 [0.04677682 0.4841499  0.20018439 0.6867032 ]
 [0.06142904 0.49832204 0.21186306 0.72141899]
 [0.07175781 0.50121515 0.21767439 0.70081662]
 [0.07504234 0.48738587 0.20627249 0.71594517]]
0.072408-0.492059-0.204641-0.696594


array([0.072408, 0.492059, 0.204641, 0.696594])

week-04

In [6]:
X_init = np.array([
 [0.19144708, 0.03819337, 0.60741781, 0.41458414],
 [0.75865295, 0.53651774, 0.65600038, 0.36034155],
 [0.43834987, 0.8043397 , 0.21024527, 0.15129482],
 [0.70605083, 0.53419196, 0.26424335, 0.48208755],
 [0.83647799, 0.19360965, 0.6638927 , 0.78564888],
 [0.68343225, 0.11866264, 0.82904591, 0.56757661],
 [0.55362148, 0.66734998, 0.32380582, 0.81486975],
 [0.35235627, 0.32224153, 0.11697937, 0.47311252],
 [0.15378571, 0.72938169, 0.42259844, 0.44307417],
 [0.46344227, 0.63002451, 0.10790646, 0.9576439 ],
 [0.67749115, 0.35850951, 0.47959222, 0.07288048],
 [0.58397341, 0.14724265, 0.34809746, 0.42861465],
 [0.30688872, 0.31687813, 0.62263448, 0.09539906],
 [0.51114177, 0.817957  , 0.72871042, 0.11235362],
 [0.43893338, 0.77409176, 0.37816709, 0.93369621],
 [0.22418902, 0.84648049, 0.87948418, 0.87851568],
 [0.72526172, 0.47987049, 0.08894684, 0.75976022],
 [0.35548161, 0.63961937, 0.41761768, 0.12260384],
 [0.11987923, 0.86254031, 0.64333133, 0.84980383],
 [0.12688467, 0.15342962, 0.77016219, 0.19051811]
])

y_init = np.array([
 6.44434399e+01, 1.83013796e+01, 1.12939795e-01, 4.21089813e+00,
 2.58370525e+02, 7.84343889e+01, 5.75715369e+01, 1.09571876e+02,
 8.84799176e+00, 2.33223610e+02, 2.44230883e+01, 6.44201468e+01,
 6.34767158e+01, 7.97291299e+01, 3.55806818e+02, 1.08885962e+03,
 2.88667516e+01, 4.51815703e+01, 4.31612757e+02, 9.97233189e+00
])

X_all = np.vstack([
X_init, 
[0.232877,0.841416,0.883342,0.879464],
[0.245154,0.843081,0.898729,0.883391],
[0.237286,0.897828,0.947445,0.897134]
])

y_all = np.hstack([
y_init, 
1091.3271430129832,
1195.7279770056589,
1880.5766271350337
])


eps= 1e-20
signs = np.sign(y_all)
signs[signs == 0] = 1.0
y_trans = signs * np.log10(np.abs(y_all) + eps)


kernel = Matern(length_scale = 0.1, nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_all, y_trans)

def acquisition_ei(X, gp, y_best, xi=0.05):
    mu, sigma = gp.predict(X, return_std=True)
    sigma = sigma.reshape(-1, 1)
    mu = mu.reshape(-1, 1)
    imp = mu - y_best - xi
    Z = imp / (sigma + 1e-9)
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei.ravel()

grid_size = 100
margin = 0.03
n = 6

# Create 1D ranges
x1 = np.linspace(margin, 1-margin, n)
x2 = np.linspace(margin, 1-margin, n)
x3 = np.linspace(margin, 1-margin, n)
x4 = np.linspace(margin, 1-margin, n)

# Create 4D meshgrid
X1, X2, X3, X4 = np.meshgrid(
    x1, x2, x3, x4, 
    indexing='ij'
)

# Convert into candidate points
X_candidates = np.vstack([
    X1.ravel(),
    X2.ravel(),
    X3.ravel(),
    X4.ravel()
]).T

#bounds = [(X_all[:,i].min(), X_all[:,i].max()) for i in range(4)]
#num_candidates = 5000
#X_candidates = np.column_stack([
#    np.random.uniform(b[0], b[1], num_candidates) for b in bounds
#])

# --- 6. Compute EI across the grid ---
y_best = np.max(y_trans)
acq_values = acquisition_ei(X_candidates, gp, y_best, xi=1.0)

# --- 7. Select the next point ---
next = X_candidates[np.argmax(acq_values)]
best_ei = np.max(acq_values)

# --- 8. Display results with precision ---
print(f"[{next[0]:.6f}, {next[1]:.6f}, {next[2]:.6f}, {next[3]:.6f}]")

print(cf.format_inputdata(next))

[0.594000, 0.970000, 0.970000, 0.970000]
0.594000-0.970000-0.970000-0.970000


[0.406000, 0.970000, 0.970000, 0.970000] for 0.05 <br/>
[0.782000, 0.970000, 0.970000, 0.970000] for 1.0 <br/>
[0.782000, 0.970000, 0.970000, 0.970000] for 2.0 <br/>

Week-05

In [7]:
from sklearn.gaussian_process.kernels import ConstantKernel, WhiteKernel
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

In [8]:
X_init = np.array([
 [0.19144708, 0.03819337, 0.60741781, 0.41458414],
 [0.75865295, 0.53651774, 0.65600038, 0.36034155],
 [0.43834987, 0.8043397 , 0.21024527, 0.15129482],
 [0.70605083, 0.53419196, 0.26424335, 0.48208755],
 [0.83647799, 0.19360965, 0.6638927 , 0.78564888],
 [0.68343225, 0.11866264, 0.82904591, 0.56757661],
 [0.55362148, 0.66734998, 0.32380582, 0.81486975],
 [0.35235627, 0.32224153, 0.11697937, 0.47311252],
 [0.15378571, 0.72938169, 0.42259844, 0.44307417],
 [0.46344227, 0.63002451, 0.10790646, 0.9576439 ],
 [0.67749115, 0.35850951, 0.47959222, 0.07288048],
 [0.58397341, 0.14724265, 0.34809746, 0.42861465],
 [0.30688872, 0.31687813, 0.62263448, 0.09539906],
 [0.51114177, 0.817957  , 0.72871042, 0.11235362],
 [0.43893338, 0.77409176, 0.37816709, 0.93369621],
 [0.22418902, 0.84648049, 0.87948418, 0.87851568],
 [0.72526172, 0.47987049, 0.08894684, 0.75976022],
 [0.35548161, 0.63961937, 0.41761768, 0.12260384],
 [0.11987923, 0.86254031, 0.64333133, 0.84980383],
 [0.12688467, 0.15342962, 0.77016219, 0.19051811],
 [0.232877,0.841416,0.883342,0.879464], #week-01 new input
 [0.245154,0.843081,0.898729,0.883391], #week-02 new input
 [0.237286,0.897828,0.947445,0.897134], #week-03 new input
 [0.594000, 0.970000, 0.970000, 0.970000] #week-04 new input
])

y_init = np.array([
 6.44434399e+01, 1.83013796e+01, 1.12939795e-01, 4.21089813e+00,
 2.58370525e+02, 7.84343889e+01, 5.75715369e+01, 1.09571876e+02,
 8.84799176e+00, 2.33223610e+02, 2.44230883e+01, 6.44201468e+01,
 6.34767158e+01, 7.97291299e+01, 3.55806818e+02, 1.08885962e+03,
 2.88667516e+01, 4.51815703e+01, 4.31612757e+02, 9.97233189e+00,
 1091.3271430129832, #week-01 new output
 1195.7279770056589, #week-02 new output
 1880.5766271350337, #week-03 new output
 3735.477417437672 #week-04 new output
])

eps= 1e-20
signs = np.sign(y_init)
signs[signs == 0] = 1.0
y_trans = signs * np.log10(np.abs(y_init) + eps)

#kernel = Matern(length_scale = 0.1, nu=2.5)
kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=0.25, length_scale_bounds=(1e-2, 2.0), nu=2.5) \
         + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-10, 1e-1))

#gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp = GaussianProcessRegressor(
    kernel=kernel, 
    alpha=1e-8, 
    normalize_y=True,
    n_restarts_optimizer=8,
    random_state=42
    )

gp.fit(X_init, y_trans)

rf = RandomForestRegressor(
    n_estimators=600, 
    random_state=42,
    bootstrap=True,
    max_features=1.0,
    min_samples_leaf=1,
    n_jobs=-1
    )

rf.fit(X_init, y_trans)

gbm_ens=[]
for k in range(12):
    gbm = GradientBoostingRegressor(
        n_estimators=350, 
        learning_rate=0.05, 
        max_depth=3, 
        random_state=100+k,
        subsample=0.8,
        min_samples_leaf=2
        )
    gbm.fit(X_init, y_trans)
    gbm_ens.append(gbm)



def acquisition_ei(X, gp, y_best, xi=0.01):
    mu, sigma = gp.predict(X, return_std=True)
    sigma = sigma.reshape(-1, 1)
    mu = mu.reshape(-1, 1)
    imp = mu - y_best - xi
    Z = imp / (sigma + 1e-9)
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei.ravel()

grid_size = 100
margin = 0.02
n=6
x1 = np.linspace(margin, 1 - margin, n)
x2 = np.linspace(margin, 1 - margin, n)
x3 = np.linspace(margin, 1-margin, n)
x4 = np.linspace(margin, 1-margin, n)

X1, X2, X3, X4 = np.meshgrid(
    x1, x2, x3, x4,
    indexing='ij'
)

# Convert into candidate points
X_candidates = np.vstack([
    X1.ravel(),
    X2.ravel(),
    X3.ravel(),
    X4.ravel()
]).T

# --- 6. Compute EI across the grid ---
y_best = np.max(y_trans)
acq_values = acquisition_ei(X_candidates, gp, y_best, xi=0.5)

# --- 7. Select the next point ---
next_point = X_candidates[np.argmax(acq_values)]
best_ei = np.max(acq_values)

# --- 8. Display results with precision ---
print(f"Next point using ei: [{next_point[0]:.6f}, {next_point[1]:.6f}, {next_point[2]:.6f}, {next_point[3]:.6f}]")

#res_formatted = [f"{r:.6f}" for r in next_point]
#result = "-".join(res_formatted)
print(cf.format_inputdata(next_point))


Next point using ei: [0.020000, 0.020000, 0.020000, 0.980000]
0.020000-0.020000-0.020000-0.980000
